In [0]:
# ---------------------------------------------------------
# 0. Set-up Imports
# ---------------------------------------------------------

import os
import uuid
import hashlib
import logging
import sys 
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable

In [0]:
# ---------------------------------------------------------
# 0.1 Set-up Logger to Track Pipeline Activities
# ---------------------------------------------------------

def setup_logging():
    # Ensure logs directory exists before starting logging
    log_dir = '/tmp/logs'
    os.makedirs(log_dir, exist_ok=True)

    logging.basicConfig(
        level = logging.INFO,
        format = '%(asctime)s - %(levelname)s - %(message)s',
        handlers = [logging.FileHandler(f'{log_dir}/bronze_ingestion_run_{datetime.now().strftime('%Y%m%d')}log'),logging.StreamHandler(sys.stdout)]
    )

    return logging.getLogger(__name__)

In [0]:
# ---------------------------------------------------------
# 0.2 Set-up Function to Identify Correct Files
# ---------------------------------------------------------

def get_correct_files(
    logger,
    volume_path,
    volume_identifier
):

    files = []

    for vol_path, _, filenames in os.walk(volume_path):

        for file in filenames:

            if volume_identifier in file:

                full_path = os.path.join(vol_path, file)

                logger.info(f"Discovered file: {full_path}")

                files.append(full_path)

    logger.info(f"Total files discovered: {len(files)}")

    return files

In [0]:
# ---------------------------------------------------------
# 0.3 Set-up Function to Identify Already Processed Files
# ---------------------------------------------------------

def file_already_processed(
    spark,
    control_table,
    source_system,
    bronze_table,
    file_hash
):

    df = (
        spark.table(control_table)
        .filter(F.col("source_system") == source_system)
        .filter(F.col("bronze_table") == bronze_table)
        .filter(F.col("file_hash") == file_hash)
        .filter(F.col("ingestion_status") == "SUCCESS")
    )

    return df.limit(1).count() > 0

In [0]:
# ---------------------------------------------------------
# 0.4 Set-up Function to Size of File
# ---------------------------------------------------------

def get_file_size(file_path):

    return os.path.getsize(file_path)

In [0]:
# ---------------------------------------------------------
# 0.5 Set-up Function to Generate Hash for File
# ---------------------------------------------------------

def generate_file_hash(file_path):

    hash_md5 = hashlib.md5()

    with open(file_path, "rb") as f:

        for chunk in iter(lambda: f.read(4096), b""):

            hash_md5.update(chunk)

    return hash_md5.hexdigest()

In [0]:
# ---------------------------------------------------------
# 0.6 Set-up Function to Register Start of Ingestion
# ---------------------------------------------------------

def register_ingestion_start(
    spark,
    control_table,
    source_system,
    bronze_table,
    file_path,
    pipeline_run_id,
    batch_id
):

    file_size = get_file_size(file_path)

    file_hash = generate_file_hash(file_path)

    df = spark.createDataFrame(
        [(
            source_system,
            bronze_table,
            file_path,
            file_hash,
            file_size,
            batch_id,
            "PENDING",
            pipeline_run_id
        )],
        [
            "source_system",
            "bronze_table",
            "file_path",
            "file_hash",
            "file_size",
            "batch_id",
            "ingestion_status",
            "pipeline_run_id"
        ]
    ).withColumn(
        "ingestion_timestamp",
        F.current_timestamp()
    )

    (
        df.write
        .format("delta")
        .mode("append")
        .saveAsTable(control_table)
    )

In [0]:
# ---------------------------------------------------------
# 0.7 Set-up Function to Register End of Ingestion
# ---------------------------------------------------------

def register_ingestion_success(
    spark,
    control_table,
    source_system,
    bronze_table,
    file_path,
    row_count
):

    delta_table = DeltaTable.forName(
        spark,
        control_table
    )

    (
        delta_table.update(
            condition=f"""
                source_system = '{source_system}'
                AND bronze_table = '{bronze_table}'
                AND file_path = '{file_path}'
                AND ingestion_status = 'PENDING'
            """,
            set={
                "ingestion_status": "'SUCCESS'",
                "completion_timestamp": "current_timestamp()",
                "row_count": f"{row_count}"
            }
        )
    )

In [0]:
# ---------------------------------------------------------
# 0.8 Set-up Function to Register Ingestion Failure
# ---------------------------------------------------------

def register_ingestion_failure(
    spark,
    control_table,
    source_system,
    bronze_table,
    file_path,
    error_message
):

    delta_table = DeltaTable.forName(
        spark,
        control_table
    )

    (
        delta_table.update(
            condition=f"""
                source_system = '{source_system}'
                AND bronze_table = '{bronze_table}'
                AND file_path = '{file_path}'
                AND ingestion_status = 'PENDING'
            """,
            set={
                "ingestion_status": "'FAILED'",
                "completion_timestamp": "current_timestamp()",
                "error_message": F.lit(error_message)
            }
        )
    )

In [0]:
# ---------------------------------------------------------
# 0.9 Set-up Function to Convert Read Data into DataFrame
# ---------------------------------------------------------

def prepare_bronze_dataframe(
    logger,
    spark,
    file_path,
    batch_id
):

    logger.info(f"Preparing dataframe: {file_path}")

    file_size = get_file_size(file_path)

    file_hash = generate_file_hash(file_path)

    # ---------------------------------------------
    # READ SOURCE FILE
    # ---------------------------------------------

    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(file_path)
    )

    # ---------------------------------------------
    # APPLY METADATA
    # ---------------------------------------------

    df = (
        df
        .withColumn(
            "bronze_ingestion_timestamp",
            F.current_timestamp()
        )
        .withColumn(
            "bronze_source_file",
            F.lit(file_path)
        )
        .withColumn(
            "bronze_file_name",
            F.element_at(F.split(F.lit(file_path), "/"), -1)
        )
        .withColumn(
            "bronze_pipeline_run_id",
            F.expr("uuid()")
        )
        .withColumn(
            "bronze_batch_id",
            F.lit(batch_id)
        )
        .withColumn(
            "bronze_file_size",
            F.lit(file_size)
        )
        .withColumn(
            "bronze_file_hash",
            F.lit(file_hash)
        )
    )

    return df

In [0]:
def apply_bronze_metadata(
    df,
    file_path,
    batch_id,
    file_hash,
    file_size=None
):

    metadata_df = (
        df

        # -----------------------------------
        # INGESTION METADATA
        # -----------------------------------

        .withColumn(
            "bronze_ingestion_timestamp",
            F.current_timestamp()
        )

        #.withColumn(
        #    "bronze_load_date",
        #    F.current_date()
        #)

        # -----------------------------------
        # SOURCE FILE METADATA
        # -----------------------------------

        .withColumn(
            "bronze_source_file",
            F.lit(file_path)
        )

        .withColumn(
            "bronze_file_name",
            F.element_at(
                F.split(F.lit(file_path), "/"),
                -1
            )
        )

        .withColumn(
            "bronze_file_hash",
            F.lit(file_hash)
        )

        # -----------------------------------
        # PIPELINE EXECUTION METADATA
        # -----------------------------------

        .withColumn(
            "bronze_batch_id",
            F.lit(batch_id)
        )

        .withColumn(
            "bronze_pipeline_run_id",
            F.expr("uuid()")
        )
    )

    # -----------------------------------
    # OPTIONAL FILE SIZE
    # -----------------------------------

    if file_size is not None:

        metadata_df = metadata_df.withColumn(
            "bronze_file_size",
            F.lit(file_size)
        )

    return metadata_df

In [0]:
# ---------------------------------------------------------
# 0.10 Set-up Function to Append Data to Bronze Table
# ---------------------------------------------------------
def append_to_bronze_table(
    logger,
    df,
    bronze_table_name,
):

    (
        df.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(bronze_table_name)
    )

    logger.info(f"Appended to:{bronze_table_name}")